In [3]:
import collections
import re
from d2l import torch as d2l

#@save
d2l.DATA_HUB['time_machine'] = (d2l.DATA_URL + 'timemachine.txt',
                                '090b5e7e70c295757f55df93cb0a180b9691891a')

def read_time_machine():  #@save
    """将时间机器数据集加载到文本行的列表中"""
    with open(d2l.download('time_machine'), 'r') as f:
        lines = f.readlines()
    return [re.sub('[^A-Za-z]+', ' ', line).strip().lower() for line in lines]

lines = read_time_machine()
print(f'# 文本总行数: {len(lines)}')
print(lines[0])
print(lines[10])

def tokenize(lines, token='word'):  #@save
    """将文本行拆分为单词或字符词元"""
    if token == 'word':
        return [line.split() for line in lines]
    elif token == 'char':
        return [list(line) for line in lines]
    else:
        print('错误：未知词元类型：' + token)

tokens = tokenize(lines)
for i in range(11):
    print(tokens[i])

# 文本总行数: 3221
the time machine by h g wells
twinkled and his usually pale face was flushed and animated the
['the', 'time', 'machine', 'by', 'h', 'g', 'wells']
[]
[]
[]
[]
['i']
[]
[]
['the', 'time', 'traveller', 'for', 'so', 'it', 'will', 'be', 'convenient', 'to', 'speak', 'of', 'him']
['was', 'expounding', 'a', 'recondite', 'matter', 'to', 'us', 'his', 'grey', 'eyes', 'shone', 'and']
['twinkled', 'and', 'his', 'usually', 'pale', 'face', 'was', 'flushed', 'and', 'animated', 'the']


#### 随机采样 与 顺序分区
随机采样每次都随机采取一个序列，而顺序分区的操作是：把整本段内容分成 batch_size 块大块，然后让每一个处理器各读各的块。就比如：

Processor1：[0, 1, 2, ..., 16] -- [0,1,2,3,4,5] -- [6,7,8,9,10] -- ...

Processor2：[16, 17, 18, ..., 32] -- [16,17,18,19,20] -- [21,22,23,24,25] -- ...

Processor3：[33, 34, 35, ..., 48] -- [33,34..] -- [] -- ...

这意味着当我们在训练 RNN 时，处理完第一个 Batch 后，我们不需要把 RNN 脑海里的隐状态 h_t 清零。可以直接把第一个 Batch 得到的隐状态，继续传给第二个 Batch。